# 0. INICIALIZACION

In [46]:
import sys
from datetime import datetime
import pandas as pd

In [2]:
sys.path.append('../src')

In [16]:
from xml_handler.LecturaXML import LecturaXML
from LLM_interaction.LLM_interaction import LlamadaLLM

# 1. LECTURA XML

In [7]:
xml_name = "res8-4_clean.xml"
xml_path = f"C:/Users/andre/OneDrive/Escritorio/TFM/KAG/AI-KG/AI-KG Dataset/DATA/{xml_name}"

In [8]:
xml_raw = LecturaXML(xml_path)

In [10]:
abstracts = xml_raw.get_abs()

In [11]:
abstracts[0]

('arXiv:2003.07270',
 '  With the rapid advances in computing and information technologies,\ntraditional access control models have become inadequate in terms of capturing\nfine-grained, and expressive security requirements of newly emerging\napplications. An attribute-based access control (ABAC) model provides a more\nflexible approach for addressing the authorization needs of complex and dynamic\nsystems. While organizations are interested in employing newer authorization\nmodels, migrating to such models pose as a significant challenge. Many\nlarge-scale businesses need to grant authorization to their user populations\nthat are potentially distributed across disparate and heterogeneous computing\nenvironments. Each of these computing environments may have its own access\ncontrol model. The manual development of a single policy framework for an\nentire organization is tedious, costly, and error-prone.\n  In this paper, we present a methodology for automatically learning ABAC\npolicy 

# 2. DEFINICION PROMPT

In [45]:
instrucciones = """
Tienes que extraer entidades del texto que te voy a pasar a continuación siguiendo estas reglas:
1. Extrae solo las entidades relevantes con el campo de la IA.
2. Cada entidad no puede contener más de 5 palabras individuales.
3. Devuelve exclusivamente estas entidades, sin explicaciones adicionales, solo una lista de palabras.
4. No pongas saltos de linea (/n). Las Entidades deben estar separadas por ;
5. El output tiene que seguir EXACTAMENTE este formato: ENTIDAD1;ENTIDAD2;ENTIDAD3;ENTIDAD4...


Texto a analizar: 

"""

In [32]:
abs_id = abstracts[0][0]
abstract = abstracts[0][1]
abstract

'  With the rapid advances in computing and information technologies,\ntraditional access control models have become inadequate in terms of capturing\nfine-grained, and expressive security requirements of newly emerging\napplications. An attribute-based access control (ABAC) model provides a more\nflexible approach for addressing the authorization needs of complex and dynamic\nsystems. While organizations are interested in employing newer authorization\nmodels, migrating to such models pose as a significant challenge. Many\nlarge-scale businesses need to grant authorization to their user populations\nthat are potentially distributed across disparate and heterogeneous computing\nenvironments. Each of these computing environments may have its own access\ncontrol model. The manual development of a single policy framework for an\nentire organization is tedious, costly, and error-prone.\n  In this paper, we present a methodology for automatically learning ABAC\npolicy rules from access logs

In [68]:
len(abstracts[0:100])

100

In [33]:
prompt_final = f"{instrucciones}{abstract}"
prompt_final

'\nTienes que extraer entidades del texto que te voy a pasar a continuación siguiendo estas reglas:\n1. Extrae solo las entidades relevantes con el campo de la IA.\n2. Cada entidad no puede contener más de 5 palabras individuales.\n3. Devuelve exclusivamente estas entidades, sin explicaciones adicionales, solo una lista de palabras.\n4. No pongas saltos de linea (/n). Las Entidades deben estar separadas por ;\n5. El output tiene que seguir EXACTAMENTE este esquema: ENTIDAD1;ENTIDAD2;ENTIDAD3;ENTIDAD4...\n\n\nTexto a analizar: \n\n  With the rapid advances in computing and information technologies,\ntraditional access control models have become inadequate in terms of capturing\nfine-grained, and expressive security requirements of newly emerging\napplications. An attribute-based access control (ABAC) model provides a more\nflexible approach for addressing the authorization needs of complex and dynamic\nsystems. While organizations are interested in employing newer authorization\nmodels,

# 3. LLAMADA MODELO

In [48]:
modelo_LLM = "phi3"

In [ ]:
respuesta = LlamadaLLM.generate(modelo_LLM, prompt_final)

In [35]:
respuesta

'```text\ncomputing;attributes-based access control model;ABAC;access logs;learning algorithm;policy rules;patterns;entropy optimization technique;automatic learning approach;authorization needs;complex systems;rule pruning algorithms;policy refinement algorithms;unsupervised learning;migration challenges;large-scale businesses;user populations\n```'

# 4. ACUMULAR RESULTADOS EN TABLA

In [41]:
output_df = pd.DataFrame(columns=["id", "entidades"])

In [42]:
output_df

,id,entidades


In [43]:
output_df.loc[len(output_df)] = [abs_id, respuesta]

In [44]:
output_df

,id,entidades
0,arXiv:2003.07270,```text\ncomputing;attributes-based access con...


# 5. GUARDAR RESULTADOS

In [50]:
nombre_archivo = "prueba_1"

## 5.1 GUARDAR .txt CON METADATOS

In [59]:
xml_utilizados = xml_name
modelo_utilizado = modelo_LLM
prompt_utilizado = instrucciones

ahora = datetime.now()
fecha_hora = ahora.strftime("%Y-%m-%d %H:%M:%S")

nombre_archivo = ahora.strftime("registro_%Y-%m-%d_%H%M.txt")


with open(nombre_archivo, "w", encoding="utf-8") as f:
    f.write(f"fecha_ejecucion: {fecha_hora}\n\n")
    f.write(f"xml_utilizados: {xml_utilizados}\n\n")
    f.write(f"modelo_utilizado: {modelo_utilizado}\n\n")
    f.write(f'prompt_utilizado: \n"{prompt_utilizado}"')

In [61]:
ahora = datetime.now()

In [63]:
fecha_hora = ahora.strftime("%Y-%m-%d_%H:%M")

In [64]:
fecha_hora

'2026-04-16_17:28'

## 5.2 GUARDAR  CSV

In [60]:
path = "C:/Users/andre/OneDrive/Escritorio/TFM/KAG/AI-KG/AI-KG Dataset/DATA/"
out_file = ahora.strftime("OUTPUT_%Y-%m-%d_%H%M.csv")
output_df.to_csv(out_file, index=False)